In [1]:
import sys
print(sys.executable)

c:\Users\santi\venvs\supply-chain-inventory\Scripts\python.exe


In [2]:
from pathlib import Path
import pandas as pd

In [3]:
project_path = Path.cwd().parent
raw_path = project_path / "data" / "raw"

orders = pd.read_csv(raw_path / "Sales_Orders.csv")

In [4]:
orders.shape

(73595, 16)

In [5]:
orders.info()


<class 'pandas.DataFrame'>
RangeIndex: 73595 entries, 0 to 73594
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   OrderID                      73595 non-null  int64  
 1   CustomerID                   73595 non-null  int64  
 2   SalespersonPersonID          73595 non-null  int64  
 3   PickedByPersonID             62972 non-null  float64
 4   ContactPersonID              73595 non-null  int64  
 5   BackorderOrderID             7538 non-null   float64
 6   OrderDate                    73595 non-null  str    
 7   ExpectedDeliveryDate         73595 non-null  str    
 8   CustomerPurchaseOrderNumber  73595 non-null  int64  
 9   IsUndersupplyBackordered     73595 non-null  bool   
 10  Comments                     0 non-null      float64
 11  DeliveryInstructions         0 non-null      float64
 12  InternalComments             0 non-null      float64
 13  PickingCompletedWhen       

In [6]:
print("Duplicados:", orders.duplicated().sum())

print("\nPorcentaje de valores faltantes:")
print((orders.isna().mean() * 100).round(2))

Duplicados: 0

Porcentaje de valores faltantes:
OrderID                          0.00
CustomerID                       0.00
SalespersonPersonID              0.00
PickedByPersonID                14.43
ContactPersonID                  0.00
BackorderOrderID                89.76
OrderDate                        0.00
ExpectedDeliveryDate             0.00
CustomerPurchaseOrderNumber      0.00
IsUndersupplyBackordered         0.00
Comments                       100.00
DeliveryInstructions           100.00
InternalComments               100.00
PickingCompletedWhen             4.19
LastEditedBy                     0.00
LastEditedWhen                   0.00
dtype: float64


In [7]:
csv_files = list(raw_path.glob("*.csv"))

profile = []

for file in csv_files:
    df = pd.read_csv(file)

    profile.append({
        "table": file.stem,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicates": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum()
    })

profile_df = pd.DataFrame(profile)

profile_df

,table,rows,columns,duplicates,missing_values
0,Application_Cities,37940,8,0,11048
1,Application_Countries,190,14,0,0
2,Application_DeliveryMethods,10,5,0,0
3,Application_StateProvinces,53,10,0,1
4,Purchasing_PurchaseOrderLines,8367,12,0,9
5,Purchasing_PurchaseOrders,2074,12,0,4148
6,Purchasing_SupplierCategories,9,5,0,0
7,Purchasing_Suppliers,13,29,0,19
8,Sales_Customers,663,31,0,2250
9,Sales_InvoiceLines,228265,13,0,0


In [8]:
missing_profile = []

for file in csv_files:
    df = pd.read_csv(file)

    missing_pct = (df.isna().mean() * 100).round(2)

    for column, pct in missing_pct.items():
        if pct > 0:
            missing_profile.append({
                "table": file.stem,
                "column": column,
                "missing_pct": pct
            })

missing_df = pd.DataFrame(missing_profile)

missing_df

,table,column,missing_pct
0,Application_Cities,LatestRecordedPopulation,29.12
1,Application_StateProvinces,Border,1.89
2,Purchasing_PurchaseOrderLines,LastReceiptDate,0.11
3,Purchasing_PurchaseOrders,Comments,100.00
4,Purchasing_PurchaseOrders,InternalComments,100.00
5,Purchasing_Suppliers,DeliveryMethodID,30.77
6,Purchasing_Suppliers,InternalComments,84.62
7,Purchasing_Suppliers,DeliveryAddressLine1,30.77
8,Sales_Customers,BuyingGroupID,39.37
9,Sales_Customers,AlternateContactPersonID,39.37


In [9]:
processed_path = project_path / "data" / "processed"
processed_path.mkdir(exist_ok=True)

for file in csv_files:
    df = pd.read_csv(file)

    # Eliminar columnas completamente vacías
    df = df.dropna(axis=1, how="all")

    # Convertir columnas de fechas y horarios
    date_columns = [
        col for col in df.columns
        if "Date" in col or "When" in col
    ]

    for col in date_columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    # Exportar tabla limpia
    output_file = processed_path / file.name
    df.to_csv(output_file, index=False)

    print(f"{file.stem}: {df.shape[0]} filas | {df.shape[1]} columnas")

Application_Cities: 37940 filas | 8 columnas
Application_Countries: 190 filas | 14 columnas
Application_DeliveryMethods: 10 filas | 5 columnas
Application_StateProvinces: 53 filas | 10 columnas
Purchasing_PurchaseOrderLines: 8367 filas | 12 columnas
Purchasing_PurchaseOrders: 2074 filas | 10 columnas
Purchasing_SupplierCategories: 9 filas | 5 columnas
Purchasing_Suppliers: 13 filas | 29 columnas
Sales_Customers: 663 filas | 29 columnas
Sales_InvoiceLines: 228265 filas | 13 columnas
Sales_Invoices: 70510 filas | 20 columnas
Sales_OrderLines: 231412 filas | 12 columnas
Sales_Orders: 73595 filas | 13 columnas
Warehouse_StockGroups: 10 filas | 5 columnas
Warehouse_StockItemHoldings: 227 filas | 9 columnas
Warehouse_StockItems: 227 filas | 23 columnas
Warehouse_StockItemStockGroups: 442 filas | 5 columnas
Warehouse_StockItemTransactions: 236667 filas | 11 columnas


In [10]:
validation = []

for file in csv_files:
    raw_df = pd.read_csv(file)

    processed_file = processed_path / file.name
    processed_df = pd.read_csv(processed_file)

    validation.append({
        "table": file.stem,
        "raw_rows": raw_df.shape[0],
        "processed_rows": processed_df.shape[0],
        "rows_match": raw_df.shape[0] == processed_df.shape[0],
        "empty_columns": processed_df.isna().all().sum()
    })

validation_df = pd.DataFrame(validation)

validation_df

,table,raw_rows,processed_rows,rows_match,empty_columns
0,Application_Cities,37940,37940,True,0
1,Application_Countries,190,190,True,0
2,Application_DeliveryMethods,10,10,True,0
3,Application_StateProvinces,53,53,True,0
4,Purchasing_PurchaseOrderLines,8367,8367,True,0
5,Purchasing_PurchaseOrders,2074,2074,True,0
6,Purchasing_SupplierCategories,9,9,True,0
7,Purchasing_Suppliers,13,13,True,0
8,Sales_Customers,663,663,True,0
9,Sales_InvoiceLines,228265,228265,True,0


In [11]:
## Data Cleaning Summary

- 18 raw tables were profiled and processed.
- No duplicate records were identified.
- Columns containing 100% missing values were removed.
- Date and datetime fields were standardized.
- Business-relevant missing values were preserved.
- Row counts were validated between RAW and processed datasets.
- All 18 tables retained 100% of their original records.

The processed datasets are stored in `data/processed/` and are ready for PostgreSQL.

SyntaxError: invalid syntax (206944635.py, line 3)

In [12]:
pip install sqlalchemy "psycopg[binary]"


   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ------------------- -------------------- 1.0/2.2 MB 6.2 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 6.8 MB/s  0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   -------------- ------------------------- 1.3/3.7 MB 7.0 MB/s eta 0:00:01
   ------------------------------- -------- 2.9/3.7 MB 7.5 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 7.2 MB/s  0:00:00

   ---------- ----------------------------- 1/4 [psycopg]
   ---------- ----------------------------- 1/4 [psycopg]
   ---------- ----------------------------- 1/4 [psycopg]
   -------------------- ------------------- 2/4 [greenlet]
   ------------------------------ --------- 3/4 [sqlalchemy]
   ------------------------------ --------- 3/4 [sqlalchemy]
   ------------------------------ --------- 3/4 [sqlalchemy]
   ------------------------------ --------- 3/4 [sqlalchemy]
   -------

In [14]:
from sqlalchemy import create_engine, text, URL
from getpass import getpass

password = getpass("Contraseña de PostgreSQL: ")

url = URL.create(
    "postgresql+psycopg",
    username="postgres",
    password=password,
    host="localhost",
    port=5432,
    database="supply_chain_inventory"
)

engine = create_engine(url)

with engine.connect() as connection:
    result = connection.execute(text("SELECT current_database();"))
    print(result.scalar())

supply_chain_inventory


In [15]:
processed_path = project_path / "data" / "processed"

processed_files = list(processed_path.glob("*.csv"))

for file in processed_files:
    df = pd.read_csv(file)

    table_name = file.stem.lower()

    print(f"Cargando {table_name}...")

    df.to_sql(
        name=table_name,
        con=engine,
        schema="staging",
        if_exists="replace",
        index=False,
        chunksize=5000
    )

    print(f"  -> {len(df):,} filas cargadas")

print("\nCarga a PostgreSQL completada.")

Cargando application_cities...
  -> 37,940 filas cargadas
Cargando application_countries...
  -> 190 filas cargadas
Cargando application_deliverymethods...
  -> 10 filas cargadas
Cargando application_stateprovinces...
  -> 53 filas cargadas
Cargando purchasing_purchaseorderlines...
  -> 8,367 filas cargadas
Cargando purchasing_purchaseorders...
  -> 2,074 filas cargadas
Cargando purchasing_suppliercategories...
  -> 9 filas cargadas
Cargando purchasing_suppliers...
  -> 13 filas cargadas
Cargando sales_customers...
  -> 663 filas cargadas
Cargando sales_invoicelines...
  -> 228,265 filas cargadas
Cargando sales_invoices...
  -> 70,510 filas cargadas
Cargando sales_orderlines...
  -> 231,412 filas cargadas
Cargando sales_orders...
  -> 73,595 filas cargadas
Cargando warehouse_stockgroups...
  -> 10 filas cargadas
Cargando warehouse_stockitemholdings...
  -> 227 filas cargadas
Cargando warehouse_stockitems...
  -> 227 filas cargadas
Cargando warehouse_stockitemstockgroups...
  -> 442 fil